In [ ]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import PokerEngine.poker_engine as pe

USE_MPS = False

# Define the device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif USE_MPS and torch.backends.mps.is_available(): # For Apple Silicon GPUs
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
torch.__version__

In [ ]:
from ReinforcementLearningEnvironment.helper import (N_ACTIONS, encode_observation)
from Models.GRUModel import GRUModel
from Models.MLPModel import MLPModel
from Models.LSTMModel import LSTMModel
from Models.TransformerQModel import TransformerQModel

from ReinforcementLearningEnvironment.Poker import train_poker_dqn
from ReinforcementLearningEnvironment.TrainingEnvironmentVariables import CSV_PATH
from ReinforcementLearningEnvironment.helper import (
    moving_average, make_13x13, plot_heatmap, 
    plot_pocket_ev_curves, plot_pocket_ev_vs_sharpe_scatter,
    pocket_win_by_street, first_action_dist, pocket_aggression_table,bluff_proxy_table
    )

In [ ]:
input_features, output_features= 151, N_ACTIONS

engine = pe.GameEngine(numPlayers=4, startingMoney=200, big_blind=2, small_blind=1)
engine.reset_tournament(200)   # if you implemented this
engine.start_hand()

obs = engine.getObservation(0)
state = encode_observation(obs, big_blind=2)
# print("state len =", len(state))


model_0 = MLPModel(input_features=input_features, output_features=output_features, hidden=(512, 1024, 2048, 1024, 512))
model_1 = LSTMModel(input_dim=input_features, output_dim=output_features)
model_2 = GRUModel(input_dim=input_features,output_dim=output_features)
model_3 = TransformerQModel(input_size=input_features,num_actions=output_features)

models = [
    model_0,
    model_1,
    model_2,
    model_3,
]

In [ ]:

MA_WINDOW = 50  # tune (e.g., 50, 200, 500)

STARTING_AMOUNT = 10000
BIG_BLIND = 2
SMALL_BLIND = 1
MAX_HANDS = 10

agents, history = train_poker_dqn(
    models=models,
    n_players=len(models),
    starting_stack=STARTING_AMOUNT,
    big_blind=BIG_BLIND,
    small_blind=SMALL_BLIND,
    max_hands=MAX_HANDS,
    device= "cpu",
    ma_window=MA_WINDOW,
    agent_kwargs=[{}, {"sequence": True}, {"sequence": True}, {"sequence": True}],  # the hand so far for LSTM, GRU, Transformer
)



In [ ]:

df = pd.read_csv(CSV_PATH)
df_run = df[df["run_id"] == df["run_id"].iloc[-1]]  # last run
# df = df[df["run_id"] == "80b9c3de"]
# df = df[df["run_id"] == "ff302c58"]

pivot = pd.pivot_table(
    df,
    index="pocket",
    columns="street_end",
    values="profitable",
    aggfunc=["count", "mean"]
)
print(df.columns)
print(df.head())
print(pivot.sort_values(("mean", 3), ascending=False))


history = {
    "ma_bb": {}
}

for player_id, g in df.groupby("player_id"):
    deltas = g["bb_delta"].to_numpy()
    history["ma_bb"][player_id] = moving_average(
        deltas, window=MA_WINDOW
    ).tolist()

In [ ]:
LAST_X_GAMES_FOR_MEAN = 100

plt.figure(figsize=(10,5))
for i in range(len(models)):
    ma = np.array(history["ma_bb"][i], dtype=np.float32)
    x = np.arange(len(ma)) + (MA_WINDOW - 1)  # align conv-valid output
    plt.plot(x, ma, label=f"P{i} - ")
    print(
        ma[-LAST_X_GAMES_FOR_MEAN:].mean(), 
        ma[-LAST_X_GAMES_FOR_MEAN:].var(),
        (ma[-LAST_X_GAMES_FOR_MEAN:].mean() - STARTING_AMOUNT) / ma[-LAST_X_GAMES_FOR_MEAN:].std()
        )

# Optional: show where tournament resets happened
# for r in history["resets"]:
#     plt.axvline(r, linewidth=1, linestyle="--", alpha=0.3)

plt.xlabel("Hand")
plt.ylabel(f"Moving Avg BB/hand (window={MA_WINDOW})")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:


df = pd.read_csv(CSV_PATH)
print("DF count: ", df.shape)
df_run = df[df["run_id"] == df["run_id"].iloc[-1]]  # last run
# df_run = df[df["run_id"] == "ff302c58"]  # last run
# df_run = df  # no filter
# df_run = df[df["run_id"] == "80b9c3de"] 

print("DF run shape:", df_run.shape)

grid_ev = make_13x13(df_run, value_col="bb_delta", agg="mean", min_samples=30)
plot_heatmap(grid_ev, "Pocket EV (mean BB/hand)")

grid_wr = make_13x13(df_run.assign(tmp=df_run["bb_delta"]), value_col="bb_delta", agg="winrate", min_samples=30)
plot_heatmap(grid_wr, "Pocket Winrate (P(bb_delta>0))")




In [ ]:
stats = ["profitable", "bb_raised", "aggressive", "raises_made", "folded", "raises_faced"]
"""'run_id', 'hand_id', 'player_id', 'pocket', 'street_end',
    'street_end_name', 'bb_delta', 'profitable', 'went_showdown',
    'non_showdown_win', 'first_action', 'first_action_name', 'bb_raised',
    'raises_made', 'raises_faced', 'aggressive'
"""
for stat in stats:
    grid_ev = make_13x13(df_run, value_col=stat, agg="mean", min_samples=30)
    plot_heatmap(grid_ev, f"Pocket EV (mean {stat})")
    grid_wr = make_13x13(df_run.assign(tmp=df_run[stat]), value_col=stat, agg="winrate", min_samples=30)
    plot_heatmap(grid_wr, f"Pocket Winrate (P({stat}>0))")

In [ ]:

# plot_pocket_ev_curve(df_run, "AKs")
# plot_pocket_ev_curve(df_run, "72o")
# plot_pocket_ev_curve(df_run, "TT")
plot_pocket_ev_curves(df=df_run, pockets=("AKs", "72o", "TT", "QQ", "AA", "KK", "KQo"), max_points=5000)


In [ ]:
stats = plot_pocket_ev_vs_sharpe_scatter(
    df_run,
    min_hands=300,
    use_excess=False,
    annualize_like=False,
    clip_sharpe=(-5, 5),
    label_top_k=20,
)
print(stats.head(20))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

offsuit = stats[stats["p169"].str.endswith("o")]
suited  = stats[stats["p169"].str.endswith("s")]
pairs  = stats[(~stats["p169"].str.endswith("s")) & (~stats["p169"].str.endswith("o"))]

xComponent = "n"
yComponent = "ev"

plt.scatter(offsuit[xComponent], offsuit[yComponent], label="Offsuit")
plt.scatter(suited[xComponent],  suited[yComponent],  label="Suited")
plt.scatter(pairs[xComponent],  pairs[yComponent],  label="Pairs")
plt.xlabel("Number of occurrences (n)")
plt.ylabel("Expected Value (ev)")
plt.title("Number of Occurrences vs Expected Value")
plt.legend()

def label_extremes(df, label_prefix="", x=xComponent, y=yComponent,
                   top_ev=2, low_ev=2, top_n=2, low_n=2,
                   text_offset=(5, 5)):
    # indices to label (union of criteria)
    idx = set()

    idx.update(df.nlargest(top_ev, y).index)
    idx.update(df.nsmallest(low_ev, y).index)
    idx.update(df.nlargest(top_n, x).index)
    idx.update(df.nsmallest(low_n, x).index)

    # annotate
    for i in idx:
        plt.annotate(
            df.loc[i, "p169"],
            (df.loc[i, x], df.loc[i, y]),
            textcoords="offset points",
            xytext=text_offset,
            ha="left",
            fontsize=9,
        )

label_extremes(offsuit, text_offset=(6, 6))
label_extremes(suited,  text_offset=(6, -10))  
label_extremes(pairs,  text_offset=(6, -10))  

plt.show()

In [ ]:

for pid in range(4):
    print("P", pid)
    print(first_action_dist(df_run, pid))
    print()

p0 = pocket_aggression_table(df_run, 0)
p1 = pocket_aggression_table(df_run, 1)

In [ ]:

# compare same pockets
merged = p0.merge(p1, on="p169", suffixes=("_p0","_p1"))
merged["ev_diff"] = merged["ev_p0"] - merged["ev_p1"]
merged["agg_diff"] = merged["agg_rate_p0"] - merged["agg_rate_p1"]
print(merged.sort_values("ev_diff", ascending=False).head(30))

grid0 = make_13x13(df_run[df_run.player_id==0], value_col="bb_delta", agg="mean", min_samples=30)
grid1 = make_13x13(df_run[df_run.player_id==1], value_col="bb_delta", agg="mean", min_samples=30)
plot_heatmap(grid0 - grid1, "EV heatmap diff: P0 - P1")



In [ ]:

bluffs_p0 = bluff_proxy_table(df_run, 0, min_samples=50)
print(bluffs_p0.head(30))


In [ ]:
print(bluffs_p0["n"].mean(), bluffs_p0["n"].quantile(0.25), bluffs_p0["n"].quantile(0.75))
print(bluffs_p0["n"].min(), bluffs_p0["n"].max())
bluffs_p0.sort_values("ev", ascending=False).head(10)

In [ ]:



print(pocket_win_by_street(df_run, player_id=0, street_end=0).head(30))  # preflop-only
print(pocket_win_by_street(df_run, player_id=0, street_end=3).head(30))  # river

